### Using a Remote MCP Server as Tools

In this section we connect to the **Open Web Search MCP server** deployed on Azure Container Apps (see `aca_mcp_server.tf`) using [`langchain-mcp-adapters`](https://github.com/langchain-ai/langchain-mcp-adapters). The adapter discovers the server's tools automatically and makes them available as LangChain tools.

Github page for Open Web Search: https://github.com/Aas-ee/open-webSearch

In [3]:
aca_gemma4_31b_it_a100_fqdn = ! terraform output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

aca_mcp_server_fqdn = ! terraform output -raw aca_mcp_server_fqdn
aca_mcp_server_fqdn = aca_mcp_server_fqdn.n
print("MCP Server Endpoint:", aca_mcp_server_fqdn)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
MCP Server Endpoint: mcp-server.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io


### Connect to the Remote MCP Server and Discover Tools

Use `MultiServerMCPClient` to connect to the MCP server over **Streamable HTTP** transport. The client automatically discovers all tools the server exposes.

In [107]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY",
    model="google/gemma-4-31B-it",
    streaming=True,
    max_completion_tokens= 4096 # 8736 # 131072 # 512
)

In [85]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(
    base_url="https://foundry-555.services.ai.azure.com/openai/v1",
    api_key="FpKIhVY2HyQc3oT9gR7dyH4p2MKAva31WEukC7CQmD81TCivL2E4JQQJ99CDACfhMk5XJ3w3AAAAACOGxtMj",
    model="gpt-5.4", # "Kimi-K2.6",
    streaming=True,
    # max_completion_tokens= 4096 # 8736 # 131072 # 512
)

In [109]:
response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I am a large language model, trained by Google.

If you think of me as a digital assistant or a creative collaborator, that’s a good way to put it. I don’t have a physical body, personal feelings, or a life story, but I have been trained on a massive amount of text data, which allows me to process information and communicate in a human-like way.

Here is a breakdown of what I can do and how I function:

### 🛠️ What I can do
*   **Answer Questions:** From complex scientific concepts to "how-to" guides or quick trivia.
*   **Write and Create:** I can draft emails, essays, poems, scripts, stories, and more.
*   **Code and Technical Work:** I can write code in various languages, debug errors, and explain technical documentation.
*   **Summarize:** I can take long articles or documents and boil them down to the key points.
*   **Translate:** I can communicate and translate across dozens of different languages.
*   **Brainstorm:** If you're stuck on a project, I can help generate ideas, outl

In [6]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the remote MCP server over Streamable HTTP
mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_fqdn}/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

# Create an agent with the self-hosted LLM + MCP tools
agent_with_mcp = create_agent(model, mcp_tools_web_search)

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


In [100]:
# Invoke a specific tool
import json

# Find the tool called "search" by name
search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

result = await search_tool.ainvoke({"query": "Azure Container Apps news", "limit": 3, "engines": ["duckduckgo"]})

# result is already a list — extract the "text" field and parse it
data = json.loads(result[0]["text"])
print(json.dumps(data, indent=2))

{
  "query": "Azure Container Apps news",
  "engines": [
    "duckduckgo"
  ],
  "totalResults": 3,
  "results": [
    {
      "title": "What&#x27;s new in Azure Container Apps at Build&#x27;25 | Microsoft Community Hub",
      "url": "https://techcommunity.microsoft.com/blog/appsonazureblog/whats-new-in-azure-container-apps-at-build25/4414891",
      "description": "<b>Azure</b> <b>Container</b> <b>Apps</b> is a fully managed serverless <b>container</b> service that runs microservices and containerized applications on <b>Azure</b>. It provides built-in autoscaling, including scale to zero, and offers simplified developer experience with support for multiple programming languages and frameworks, including special features built for .NET and Java. <b>Container</b> <b>Apps</b> also provides many advanced ...",
      "source": "techcommunity.microsoft.com",
      "engine": "duckduckgo"
    },
    {
      "title": "What&#x27;s new in Azure Container Apps - Azure Container Apps",
      "url

In [101]:
# Extract first URL from search results
first_url = data["results"][0]["url"]
print("First URL:", first_url)

# Find the tool called "search" by name
fetch_web_content_tool = next(t for t in mcp_tools_web_search if t.name == "fetchWebContent")

# Fetch the page content using the fetch tool
result = await fetch_web_content_tool.ainvoke({"url": first_url, "maxChars": 30000})

# Parse and display the fetched content
web_content_json = json.loads(result[0]["text"])
print(json.dumps(web_content_json, indent=2))

First URL: https://techcommunity.microsoft.com/blog/appsonazureblog/whats-new-in-azure-container-apps-at-build25/4414891
{
  "url": "https://techcommunity.microsoft.com/blog/appsonazureblog/whats-new-in-azure-container-apps-at-build25/4414891",
  "finalUrl": "https://techcommunity.microsoft.com/blog/appsonazureblog/whats-new-in-azure-container-apps-at-build25/4414891",
  "contentType": "text/html; charset=utf-8",
  "title": "What's new in Azure Container Apps at Build'25 | Microsoft Community Hub",
  "retrievalMethod": "request",
  "truncated": false,
  "content": "Blog PostApps on Azure Blog 6 MIN READWhat's new in Azure Container Apps at Build'25vyomnagraniMicrosoftMay 19, 2025Azure Container Apps is a fully managed serverless container service that runs microservices and containerized applications on Azure. It provides built-in autoscaling, including scale to zero, and offers simplified developer experience with support for multiple programming languages and frameworks, including sp

In [102]:
from markdownify import markdownify

markdownify(html=web_content_json["content"])

"Blog PostApps on Azure Blog 6 MIN READWhat's new in Azure Container Apps at Build'25vyomnagraniMicrosoftMay 19, 2025Azure Container Apps is a fully managed serverless container service that runs microservices and containerized applications on Azure. It provides built-in autoscaling, including scale to zero, and offers simplified developer experience with support for multiple programming languages and frameworks, including special features built for .NET and Java. Container Apps also provides many advanced networking and monitoring capabilities, offering seamless deployment and management of containerized applications without the need to manage underlying infrastructure.\nFollowing the features announced at Ignite’24, we've continued to innovate and enhance Azure Container Apps. We announced the general availability of Serverless GPUs, enabling seamless AI workloads with automatic scaling, optimized cold start, per-second billing, and reduced operational overhead. We added a preview of

In [103]:
from IPython.display import display, Markdown

display(Markdown(web_content_json["content"]))  # if fetch_data is a string
# or if it's a dict with a content field:
# display(Markdown(fetch_data["content"]))

Blog PostApps on Azure Blog 6 MIN READWhat's new in Azure Container Apps at Build'25vyomnagraniMicrosoftMay 19, 2025Azure Container Apps is a fully managed serverless container service that runs microservices and containerized applications on Azure. It provides built-in autoscaling, including scale to zero, and offers simplified developer experience with support for multiple programming languages and frameworks, including special features built for .NET and Java. Container Apps also provides many advanced networking and monitoring capabilities, offering seamless deployment and management of containerized applications without the need to manage underlying infrastructure.
Following the features announced at Ignite’24, we've continued to innovate and enhance Azure Container Apps. We announced the general availability of Serverless GPUs, enabling seamless AI workloads with automatic scaling, optimized cold start, per-second billing, and reduced operational overhead. We added a preview of JavaScript code interpreter support for Dynamic Sessions for applications that require the execution of potentially malicious JavaScript code, such as code provided by end users. Furthermore, we partnered with Aqua Security to enhance the security of Azure Container Apps, offering comprehensive image scanning, runtime protection, and supply chain security.
These advancements ensure that Azure Container Apps remains a trusted platform for running scalable, secure, and resilient containerized applications. The features we're announcing at Build’25 for new Serverless GPUs integrations, and many new networking and observability features that Enterprises care about further deepens this commitment.
Running AI workloads on Azure Container Apps
Azure Container Apps efficiently supports AI workloads with features like serverless GPUs with NVIDIA NIM integration, dynamic sessions with Hyper-V isolation, and integrations for enhanced performance and scalability. We are furthering this feature set by announcing new capabilities and integrations for Azure Container Apps.Deploy Foundry Models on Serverless GPUs for InferencingAzure Container Apps now provides an integration with Foundry Models, which allows you to deploy ready-to-use AI models directly during container app creation. This integration supports serverless APIs with pay-as-you-go billing and managed compute with pay-per-GPU pricing, providing flexibility in deploying Foundry models.

Announcing General Availability of Dedicated GPUsDedicated GPUs in Azure Container Apps are now generally available, and simplify AI application development and deployment by reducing management overhead. It offers built-in support for key components like the latest CUDA driver, turnkey networking, and security features, allowing you to focus on your AI application code.Early access to Serverless GPU in Dynamic SessionsServerless GPU in Azure Container Apps Dynamic Sessions in now available as an early access feature, enable running untrusted AI-generated code at scale within compute sandboxes protected by Hyper-V isolation. This feature supports a GPU-powered Python code interpreter to better handle AI workloads. Microsoft Dev Box offers an integration with Serverless GPU in Dynamic Sessions through the Dev Box GPU Shell feature.

Advanced Networking capabilities
Azure Container Apps offers many networking capabilities including custom VNet integration, private endpoints, user-defined routes, NAT Gateway support, and peer-to-peer encryption. We are extending these capabilities by offering new controls and features to support more nuanced network architectures.Announcing General Availability of Private EndpointsPrivate Endpoints for Azure Container Apps, now generally available, allows customers to connect to their Container Apps environment using a private IP address in their Azure Virtual Network. This eliminates exposure to the public internet and secures access to their applications. Additionally, customers can connect directly from Azure Front Door to their workload profile environments over a private link instead of the public internet.New Premium Ingress capabilitiesThe new premium ingress feature in Azure Container Apps allows for customizable ingress scaling, enabling better handling of higher demand workloads like large performance tests. It introduces environment-level ingress configuration options, including termination grace period, idle request timeout, and header count.Announcing Public Preview of rule-based routingWe are adding a rule-based routing feature in Azure Container Apps that allows you to direct incoming HTTP traffic to different apps within your Container Apps environment based on the requested host name or path. This simplifies your architecture for microservice applications, A/B testing, blue-green deployments, and more, without needing a separate reverse proxy.
Observability and debugging capabilities
Azure Container Apps provides several built-in observability features that give you a holistic view of your container app’s health throughout its application lifecycle, and help you monitor and diagnose the state of your app to improve performance and respond to trends and critical problems. We are extending these existing capabilities by introducing new observability and debugging features.Announcing General Availability of Open Telemetry CollectorThe OpenTelemetry agent in Azure Container Apps is now generally available, allowing developers to use open-source standards to send app data without setting up the collector themselves. The managed agent collects and exports telemetry data to various endpoints, including Azure Monitor Application Insights, Datadog, and any generic OTLP-configured endpoint.Announcing General Availability of Aspire dashboardThe .NET 8’s Aspire dashboard in Azure Container Apps is now generally available, providing live data about your project and containers in the cloud to evaluate performance and debug errors with comprehensive logs, metrics, and traces. In addition, we now support the newest version of Aspire (v9.2), which includes new visualization features, the ability to pause/resume telemetry, and will be globally available in the coming weeks.

New Diagnose and Solve dashboardThe new Diagnose and Solve dashboard for Azure Container Apps provides a comprehensive overview of app health, performance, and resource utilization, with insights into apps, jobs, replicas, node count, and CPU usage over time. It also includes new detectors to diagnose and resolve issues such as container create failures, health probe failures, and image pull failures.Integration with Azure SRE agentAzure Container Apps integrates seamlessly with the Azure SRE agent to enhance operational efficiency and application uptime. By continuously monitoring application health and performance, the SRE agent provides valuable insights and autonomously responds to production alerts, mitigating issues with minimal intervention. This integration allows developers to leverage the SRE agent to monitor Azure Container Apps resources, from Container App Environments to Apps to Revisions to Replicas, ensuring faster troubleshooting and proactive issue resolution.
Enhanced Enterprise capabilities
In addition to these announcements, we are introducing several enhanced Enterprise features to Azure Container Apps.Announcing General Availability of Azure Container Apps on Arc-enabled KubernetesThe ability to run Azure Container Apps on your own Azure Arc-enabled Kubernetes clusters (AKS and AKS-HCI) is now generally available. This allows developers to leverage Azure Container Apps features while IT administrators maintain corporate compliance by hosting applications in hybrid environments.Announcing General Availability of Planned Maintenance in Azure Container AppsPlanned Maintenance for Azure Container Apps is now generally available, which allows you to control when non-critical updates are applied to your environment. This helps minimize downtime and impact on applications. Critical updates are applied as needed to ensure security and reliability compliance.Announcing Public Preview of workflow capabilities with Durable Task SchedulerThe new advanced pro-code workflow feature in Azure Container Apps, leveraging durable task scheduler, is now in public preview. With durable task scheduler in Container Apps, you can create reliable workflows as code, leveraging state persistence and fault-tolerant execution. These containerized workflows enhance scalability, reliability, and streamlined monitoring for administration of complex workflows.

Native Azure Functions in Azure Container AppsThe new, streamlined method for running Azure Functions natively in Azure Container Apps allows customers to leverage the full features and capabilities of Azure Container Apps while benefiting from the simplicity of auto-scaling provided by Azure Functions. With the new native hosting model, customers can deploy Azure Functions directly onto Azure Container Apps with the same experience as deploying other containerized applications. Customers can also get the complete feature set of Azure Container Apps with this new deployment experience, including multi-revision management, easy authentication, metrics and alerting, health probes and many more.
Azure Container Apps at Build’25 conference
Also, if you're at Build, come see us at the following sessions:

Breakout 182: Better Microservices Development using Azure Container Apps
Breakout 190: Secure Next-Gen AI Apps with Azure Container Apps Serverless GPUs
Lab 341: Agentic AI Inferencing with Azure Container Apps
Community Table Talk 457: App Reliability, Azure Container Apps, & Serverless GPUs
Breakout 186: Earth’s Defense with Hera: AI Agents Battle Planet Extinction Threats
Breakout 187: Event-Driven Architectures: Serverless Apps That Slay at Scale
Breakout 201: Innovate, deploy, & optimize your apps without infrastructure hassles
Breakout 117: Use VS Code to build AI apps and agents
Breakout 185: Maximizing efficiency in cloud-native app design
Demo 544: Building Resilient Cloud-Native Microservices

Or come talk to us at the Serverless booth at the Expert Meet-up area at the Hub!
Wrapping up
As always, we invite you to visit our GitHub page for feedback, feature requests, or questions about Azure Container Apps, where you can open a new issue or up-vote existing ones. If you’re curious about what we’re working on next, check out our roadmap. We look forward to hearing from you!Updated May 18, 2025Version 1.0azure container appscontainersmodern appsserverlessCommentvyomnagraniMicrosoftJoined May 20, 2022Send MessageView ProfileApps on Azure Blog Follow this blog board to get notified when there's new activity

### Error handling for MCP Servers

Use interceptors to catch tool execution errors and implement retry logic:

In [104]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import MCPToolCallRequest
from langchain.agents import create_agent
import asyncio

async def retry_interceptor(
    request: MCPToolCallRequest,
    handler,
    max_retries: int = 3,
    delay: float = 1.0,
):
    """Retry failed tool calls with exponential backoff."""
    last_error = None
    for attempt in range(max_retries):
        try:
            return await handler(request)
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                wait_time = delay * (2 ** attempt)  # Exponential backoff
                print(f"Tool {request.name} failed (attempt {attempt + 1}), retrying in {wait_time}s...")
                await asyncio.sleep(wait_time)
    raise last_error

async def fallback_interceptor(
    request: MCPToolCallRequest,
    handler,
):
    """Return a fallback value if tool execution fails."""
    try:
        return await handler(request)
    except TimeoutError:
        return f"Tool {request.name} timed out. Please try again later."
    except ConnectionError:
        return f"Could not connect to {request.name} service. Using cached data."


### Creating an Agent with MCP Client Tools

In [110]:
# Connect to the remote MCP server over Streamable HTTP
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_fqdn}/mcp",
            "transport": "http",
        }
    },
    tool_interceptors=[retry_interceptor, fallback_interceptor]
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

# Create an agent with the self-hosted LLM + MCP tools
# model.max_completion_tokens=131072
agent_with_mcp = create_agent(model, mcp_tools_web_search)

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


### Run the Agent with MCP Tools

The agent will use the remote web search MCP tool to answer questions that require live information from the internet.

In [111]:
from langchain_core.messages import HumanMessage

async for step in agent_with_mcp.astream(
    {"messages": [HumanMessage(content="""Search the web for the latest news about Azure Container Apps in 2026.
                                          Limit search to 5 results only.
                                          Fetch and analyse web pages one by one.
                                          Use official sources only.""")]},
    stream_mode="values"
):()

step["messages"][-1].pretty_print()

================================== Ai Message ==================================

Based on the official Microsoft sources analyzed, here are the latest updates and news regarding **Azure Container Apps (ACA)** as of February 2026:

### 1. AI-Driven App Modernization (The "GitHub Copilot" Era)
The most significant shift in 2026 is the integration of **GitHub Copilot app modernization agents**. Legacy application migration to Azure Container Apps has moved from a weeks-long manual process to a matter of hours.
*   **Full Automation:** A specialized modernization agent now handles the end-to-end transition of legacy apps (e.g., .NET Framework 4.8) to modern cloud-native versions (e.g., **.NET 10**) specifically targeted for Azure Container Apps.
*   **Automated Infrastructure:** The agent now automatically generates the necessary **Bicep** files and `azure.yaml` definitions, allowing developers to deploy entire environments using the `azd up` command.
*   **Code Rewriting:** Beyond assess